In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import sys; sys.path.append('/content/drive/MyDrive/CS3268_Project/src')
import numpy as np, pandas as pd
from common import threshold_at_fpr, save_preds, PREDS
np.random.seed(42)

In [ ]:
# Group thresholds: baseline scores unchanged; one cutoff per age group, each set so 5% of that
# group's honest VAL applicants are flagged. Applied unchanged to test. No other variants.
b = pd.read_parquet(f"{PREDS}/preds_baseline.parquet")
va = b[b.month == 5]
cuts = {g: threshold_at_fpr(va[va.age_group == g].score, va[va.age_group == g].y_true, fpr=0.05) for g in (0, 1)}
print(cuts)  # expect cut_50p > cut_u50 (50-plus scores higher on average)
df = b.rename(columns={"y_true": "fraud_bool"})
print(save_preds("groupthr", df, b.score.values, cuts))

In [ ]:
# Paste this output to Claude.
p = pd.read_parquet(f"{PREDS}/preds_groupthr.parquet")
rows = []
for name, d in [("val", p[p.month == 5]), ("test", p[p.month >= 6])]:
    for grp in (0, 1, "all"):
        s = d if grp == "all" else d[d.age_group == grp]
        h, f, fl = s[s.y_true == 0], s[s.y_true == 1], s[s.decision == 1]
        rows.append(dict(split=name, group=grp, n=len(s), FPR=h.decision.mean(), recall=f.decision.mean(),
                         precision=fl.y_true.mean(), flag_rate=s.decision.mean()))
r = pd.DataFrame(rows).round(4); print(r)
for name in ("val", "test"):
    f0, f1 = r[(r.split == name) & (r.group == 0)].FPR.item(), r[(r.split == name) & (r.group == 1)].FPR.item()
    print(name, "FPR ratio (lower/higher):", round(min(f0, f1) / max(f0, f1), 3), "| flagged more:", "50-plus" if f1 > f0 else "under-50")
assert (p.score.values == b.score.values).all()  # scores identical to baseline by construction